# R&D-6. Safe in-time CUPAC — ноутбук для внутреннего контура

**Самодостаточный ноутбук.** Нужный код пакета `rnd_reports.variance_reduction` вшит ниже в ячейки — импортов `rnd_reports` нет, файл переносится во внутренний контур одним куском (нужны только `numpy / pandas / scikit-learn`).

**Как использовать во внутреннем контуре:**
1. В ячейке-конфиге укажи `INPUT_CSV` — путь к своей таблице и имена колонок (контракт ниже).
2. Ячейку-генератор синтетики (помечена ⚠️) **не запускай** — она только для проверки запускаемости в репозитории.
3. Прогони остальные ячейки: сравнение снижения дисперсии оценки ATE (A/B baseline → CUPED → safe in-time коррекция) на одной таблице.

Идея R&D-6: снижать дисперсию метрики линейной поправкой **только** по *безопасным in-time* признакам (не зависящим от назначения трита) — без смещения ATE. `true_ate` известен только для синтетики.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

## Библиотека (вшита из `rnd_reports.variance_reduction`)

In [2]:
# === Вшито из rnd_reports.variance_reduction/cuped.py ===
"""Классическая CUPED-корректировка (θ-резидуализация целевой метрики).

Извлечено из CUPAC-реализации VarWar (`autocupac.py`) без изменения численного
поведения: используется линейная поправка ``Ỹ = Y − θ·(pred − E[pred])`` с
``θ = Cov(Y, pred)/Var(pred)``. См. docs/variance_reduction_methodology.md.
"""


from typing import Optional

import numpy as np


def cuped_theta(y, prediction) -> float:
    """Коэффициент θ для CUPED-поправки.

    Вырожденный случай (почти нулевая дисперсия предсказания) → θ=0.
    Численно эквивалентно исходному коду VarWar (cov с ddof=1, var с ddof=0).
    """
    pred = np.asarray(prediction, dtype=float)
    pred_centered = pred - pred.mean()
    var = pred_centered.var()
    if var < 1e-10:
        return 0.0
    return float(np.cov(y, pred_centered)[0, 1] / var)

def cuped_adjust(y, prediction, theta: Optional[float] = None):
    """Применить CUPED-поправку к ``y`` по предсказанию ``prediction``.

    Сохраняет тип ``y`` (для ``pd.Series`` вернётся ``pd.Series``).
    """
    pred = np.asarray(prediction, dtype=float)
    pred_centered = pred - pred.mean()
    if theta is None:
        theta = cuped_theta(y, prediction)
    return y - theta * pred_centered


In [3]:
# === Вшито из rnd_reports.variance_reduction/metrics.py ===
"""Метрики снижения дисперсии для CUPED/CUPAC.

Извлечено из CUPAC-реализации VarWar (`autocupac.py`) без изменения поведения.
См. docs/variance_reduction_methodology.md.
"""




def variance_reduction_pct(y, prediction) -> float:
    """Процент снижения дисперсии ``y`` после CUPED-поправки по ``prediction``.

    Возвращает ``max(0, (1 − Var(Ỹ)/Var(Y)) · 100)``; при вырожденном
    предсказании — ``0.0`` (как в исходном коде VarWar).
    """
    theta = cuped_theta(y, prediction)
    if theta == 0.0:
        return 0.0
    y_adj = cuped_adjust(y, prediction, theta=theta)
    return max(0.0, (1 - y_adj.var() / y.var()) * 100)


In [4]:
# === Вшито из rnd_reports.variance_reduction/safe_intime_adjustment.py ===
"""Безопасная in-time коррекция метрики линейным second-stage (R&D-6, Step 5–6).

Поверх CUPAC-скорректированного исхода применяем линейную (OLS) коррекцию **только**
по допустимым in-time признакам: класс B (expert-safe) и класс C (прошедшие
balance/missingness gate). Снижает остаточную дисперсию без смещения — при условии,
что признаки безопасны (см. реестр/политику и diagnostics). Не использует treatment.

``adj = target − (ŷ − E[ŷ])``, где ``ŷ = OLS(target ~ safe_features)``.

См. docs/variance_reduction_methodology.md, docs/feature_safety_policy.md.
"""


import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression



def safe_intime_linear_adjustment(
    df: pd.DataFrame, target_col: str, safe_features: list[str]
):
    """Линейная second-stage коррекция ``target_col`` по safe in-time признакам.

    Возвращает ``(adjusted: pd.Series, info: dict)``. При пустом списке признаков
    возвращает исходный таргет без изменений.

    NaN-политика: fail-fast с понятным ``ValueError``, если в таргете или safe-признаках
    есть пропуски. Реальные датасеты могут содержать NaN — их обработку (импутация/
    исключение) добавим отдельно; пока не допускаем «молчаливого» поведения.
    """
    y = df[target_col].reset_index(drop=True)
    if not safe_features:
        return y.copy(), {"used_features": [], "variance_reduction": 0.0}

    if y.isna().any():
        raise ValueError(
            f"safe_intime_linear_adjustment: целевая колонка '{target_col}' содержит NaN"
        )
    na_cols = [c for c in safe_features if df[c].isna().any()]
    if na_cols:
        raise ValueError(
            f"safe_intime_linear_adjustment: safe-признаки содержат NaN: {na_cols}"
        )

    X = df[safe_features].reset_index(drop=True)
    model = LinearRegression().fit(X, y)
    pred = np.asarray(model.predict(X), dtype=float)
    adjusted = y - (pred - pred.mean())
    return adjusted, {
        "used_features": list(safe_features),
        "variance_reduction": variance_reduction_pct(y, pred),
    }


## 1. Данные: одна таблица из `INPUT_CSV`

Контракт таблицы: `id`, `treatment` (0/1), `target` (метрика), `pre_target` (доэкспериментальное значение метрики — предиктор CUPED) и набор безопасных in-time ковариат `safe_*`. Генератор пишет одну демонстрационную таблицу в `INPUT_CSV`; во внутреннем контуре его пропускают и подставляют реальную таблицу.

In [5]:
# --- Конфигурация прогона -------------------------------------------------------
INPUT_CSV = "data/06_safe_intime_cupac/sample.csv"  # во внутреннем контуре — своя таблица
TREATMENT_COL = "treatment"
TARGET_COL = "target"
CUPED_PREDICTOR = "pre_target"          # доэксп. метрика как предиктор CUPED
SAFE_FEATURES = ["pre_target", "safe_1", "safe_2"]  # безопасные in-time ковариаты
TRUE_ATE = None                          # реальные данные: истинный ATE неизвестен

pd.set_option('display.width', 200)
pd.set_option('display.precision', 4)

In [6]:
# +++ ЯЧЕЙКА-ГЕНЕРАТОР СИНТЕТИКИ +++
# +++ НЕ ЗАПУСКАТЬ ВО ВНУТРЕННЕМ КОНТУРЕ +++
# Синтез одной демонстрационной таблицы -> запись в INPUT_CSV. Нужна только для проверки
# запускаемости в репозитории; во внутреннем контуре используется реальная таблица.
from pathlib import Path

_SEED = 11
_rng = np.random.default_rng(_SEED)
_n = 8000
_t = _rng.integers(0, 2, _n)
_pre = _rng.normal(0.0, 1.0, _n)                 # доэксп. метрика (предиктор)
_safe1 = _rng.normal(0.0, 1.0, _n)
_safe2 = 0.5 * _pre + _rng.normal(0.0, 1.0, _n)
TRUE_ATE = 0.5                                    # истинный эффект по построению
_base = 2.0 + 1.3 * _pre + 0.7 * _safe1 + 0.4 * _safe2 + _rng.normal(0.0, 1.0, _n)
_target = _base + TRUE_ATE * _t
_df = pd.DataFrame({
    "id": np.arange(_n),
    "treatment": _t,
    "target": _target,
    "pre_target": _pre,
    "safe_1": _safe1,
    "safe_2": _safe2,
})
Path(INPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
_df.to_csv(INPUT_CSV, index=False)
print(f"синтетическая таблица записана в {INPUT_CSV}: {_n} юнитов, истинный ATE = {TRUE_ATE}")

синтетическая таблица записана в data/06_safe_intime_cupac/sample.csv: 8000 юнитов, истинный ATE = 0.5


In [7]:
# Загрузка таблицы. Во внутреннем контуре сюда приходит реальная таблица из INPUT_CSV.
df = pd.read_csv(INPUT_CSV)
print(f"таблица: {df.shape[0]} юнитов, колонки: {list(df.columns)}")

таблица: 8000 юнитов, колонки: ['id', 'treatment', 'target', 'pre_target', 'safe_1', 'safe_2']


## 2. Снижение дисперсии оценки ATE на одной таблице

Сравниваем три варианта оценки ATE на одной таблице: сырой A/B (baseline), CUPED по доэкспериментальной метрике и safe in-time линейную коррекцию по безопасным ковариатам. Главная величина — снижение дисперсии метрики (а с ней и ширины CI ATE) без смещения самого ATE.

In [8]:
def ate_ci(y, t):
    """ATE и полуширина 95%-CI (разность средних, two-sample)."""
    y = np.asarray(y, dtype=float); t = np.asarray(t)
    y1, y0 = y[t == 1], y[t == 0]
    ate = y1.mean() - y0.mean()
    half = 1.96 * np.sqrt(y1.var(ddof=1) / len(y1) + y0.var(ddof=1) / len(y0))
    return ate, half

t = df[TREATMENT_COL].to_numpy()
y = df[TARGET_COL].to_numpy()

# 1) A/B baseline (без поправок)
ate0, ci0 = ate_ci(y, t)

# 2) CUPED по доэксп. метрике
y_cuped = cuped_adjust(y, df[CUPED_PREDICTOR].to_numpy())
vr_cuped = variance_reduction_pct(pd.Series(y), df[CUPED_PREDICTOR].to_numpy())
ate1, ci1 = ate_ci(y_cuped, t)

# 3) Safe in-time линейная коррекция по безопасным ковариатам
y_safe, info = safe_intime_linear_adjustment(df, TARGET_COL, SAFE_FEATURES)
ate2, ci2 = ate_ci(y_safe.to_numpy(), t)

summary = pd.DataFrame([
    {"метод": "A/B baseline", "ATE": ate0, "CI_halfwidth": ci0, "variance_reduction_%": 0.0},
    {"метод": "CUPED", "ATE": ate1, "CI_halfwidth": ci1, "variance_reduction_%": vr_cuped},
    {"метод": "safe in-time", "ATE": ate2, "CI_halfwidth": ci2,
     "variance_reduction_%": info["variance_reduction"]},
])
if TRUE_ATE is not None:
    summary["bias_vs_true"] = summary["ATE"] - TRUE_ATE
print(summary.to_string(index=False))

       метод    ATE  CI_halfwidth  variance_reduction_%  bias_vs_true
A/B baseline 0.4168        0.0867                0.0000       -0.0832
       CUPED 0.4679        0.0565               56.5881       -0.0321
safe in-time 0.4905        0.0440               72.9188       -0.0095


## 3. Выводы (single-table)

- CUPED и safe in-time коррекция снижают дисперсию метрики (уже CI у ATE) без сдвига самой оценки ATE — на синтетике `bias_vs_true` остаётся около нуля.
- Поправка идёт **только** по безопасным in-time признакам (не зависящим от назначения трита), что и обеспечивает несмещённость; небезопасные признаки во внутренний `SAFE_FEATURES` включать нельзя.
- Сравнение цепочки методов и диагностика безопасности признаков по многим сценариям — в исследовательском `notebook.ipynb`; здесь фокус на одной таблице.